# Telecom Plan Revenue Analysis — Megaline

**Goal:** Determine which prepaid plan (Surf or Ultimate) generates more revenue per user, and test whether customers in the NY-NJ region differ significantly in revenue from other regions.

**Dataset:** 500 Megaline customers — call logs, SMS, internet sessions, and plan details across 2018.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set(style='whitegrid')

In [ ]:
users    = pd.read_csv('/datasets/megaline_users.csv')
plans    = pd.read_csv('/datasets/megaline_plans.csv')
calls    = pd.read_csv('/datasets/megaline_calls.csv')
internet = pd.read_csv('/datasets/megaline_internet.csv')
messages = pd.read_csv('/datasets/megaline_messages.csv')

print('Plans:')
display(plans)

## 2. Data Preprocessing

In [ ]:
# Parse datetime columns
users['reg_date']   = pd.to_datetime(users['reg_date'],   errors='coerce')
users['churn_date'] = pd.to_datetime(users['churn_date'], errors='coerce')
calls['call_date']       = pd.to_datetime(calls['call_date'],       errors='coerce')
messages['message_date'] = pd.to_datetime(messages['message_date'], errors='coerce')
internet['session_date'] = pd.to_datetime(internet['session_date'], errors='coerce')

# Enrich users
users['is_active']   = users['churn_date'].isna()
users['days_active'] = (users['churn_date'] - users['reg_date']).dt.days

# Extract month for aggregation
calls['month']    = calls['call_date'].dt.month
messages['month'] = messages['message_date'].dt.month
internet['month'] = internet['session_date'].dt.month

# Calls: round duration up (billing rounds to nearest minute) and drop zero-duration
calls['duration'] = np.ceil(pd.to_numeric(calls['duration'], errors='coerce'))
calls = calls[calls['duration'] > 0]

internet['mb_used'] = pd.to_numeric(internet['mb_used'], errors='coerce').fillna(0)

print(f'Calls: {len(calls):,} | Messages: {len(messages):,} | Internet sessions: {len(internet):,}')

## 3. Usage Aggregation per User per Month

In [ ]:
calls_agg    = calls.groupby(['user_id','month'])['duration'].sum().reset_index().rename(columns={'duration':'minutes_sum'})
calls_count  = calls.groupby(['user_id','month'])['id'].count().reset_index().rename(columns={'id':'calls_count'})
messages_agg = messages.groupby(['user_id','month'])['id'].count().reset_index().rename(columns={'id':'messages_count'})
internet_agg = internet.groupby(['user_id','month'])['mb_used'].sum().reset_index().rename(columns={'mb_used':'mb_used_sum'})

# Merge all usage into one DataFrame
user_usage = (calls_count
    .merge(calls_agg,    on=['user_id','month'], how='outer')
    .merge(messages_agg, on=['user_id','month'], how='outer')
    .merge(internet_agg, on=['user_id','month'], how='outer')
    .fillna(0)
)

# Attach plan details
user_usage = (user_usage
    .merge(users[['user_id','city','plan']], on='user_id', how='left')
    .merge(plans, left_on='plan', right_on='plan_name', how='left')
)

print(f'Aggregated records: {len(user_usage):,}')
user_usage[['user_id','month','plan','minutes_sum','messages_count','mb_used_sum']].head()

## 4. Revenue Calculation

In [ ]:
user_usage['gb_used']  = user_usage['mb_used_sum'] / 1024.0
user_usage['plan_gb']  = user_usage['mb_per_month_included'] / 1024.0

# Overage calculations
user_usage['extra_minutes']  = np.clip(user_usage['minutes_sum']    - user_usage['minutes_included'],  0, None)
user_usage['extra_messages'] = np.clip(user_usage['messages_count'] - user_usage['messages_included'], 0, None)
user_usage['extra_gb']       = np.ceil(np.clip(user_usage['gb_used'] - user_usage['plan_gb'], 0, None))

# Monthly revenue = base fee + overage charges
user_usage['extra_cost'] = (
    user_usage['extra_minutes']  * user_usage['usd_per_minute'] +
    user_usage['extra_messages'] * user_usage['usd_per_message'] +
    user_usage['extra_gb']       * user_usage['usd_per_gb']
)
user_usage['total_revenue'] = user_usage['usd_monthly_pay'] + user_usage['extra_cost']

user_usage[['user_id','month','plan','extra_minutes','extra_messages','extra_gb','extra_cost','total_revenue']].head(8)

## 5. Exploratory Data Analysis

### 5.1 Call Usage by Plan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=user_usage, x='plan', y='minutes_sum', ax=axes[0])
axes[0].set_title('Monthly Minutes Used by Plan')
axes[0].set_ylabel('Minutes per Month')

avg_calls = user_usage.groupby(['plan','month'])['minutes_sum'].mean().reset_index()
sns.lineplot(data=avg_calls, x='month', y='minutes_sum', hue='plan', marker='o', ax=axes[1])
axes[1].set_title('Average Monthly Minutes by Plan')
axes[1].set_ylabel('Avg Minutes')

plt.tight_layout()
plt.show()

user_usage.groupby('plan')['minutes_sum'].agg(['mean','var']).round(2)

**Finding:** Both plans show similar average minutes (~429 min/month), but Ultimate users show slightly higher variance — suggesting a wider range of usage patterns.

### 5.2 Messages & Internet Usage by Plan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=user_usage, x='plan', y='messages_count', ax=axes[0])
axes[0].set_title('Monthly Messages Sent by Plan')
axes[0].set_ylabel('Messages per Month')

sns.boxplot(data=user_usage, x='plan', y='mb_used_sum', ax=axes[1])
axes[1].set_title('Monthly Data Usage by Plan (MB)')
axes[1].set_ylabel('MB per Month')

plt.tight_layout()
plt.show()

user_usage.groupby('plan')[['messages_count','mb_used_sum']].agg(['mean','var']).round(2)

**Finding:** Ultimate users send more messages (avg 37.6 vs 31.2) and consume more data (avg 17.2 GB vs 16.6 GB). Both distributions show significant overlap, indicating usage alone does not fully explain revenue differences.

### 5.3 Revenue by Plan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=user_usage, x='plan', y='total_revenue', ax=axes[0])
axes[0].set_title('Monthly Revenue Distribution by Plan')
axes[0].set_ylabel('Revenue (USD)')

avg_rev = user_usage.groupby(['plan','month'])['total_revenue'].mean().reset_index()
sns.lineplot(data=avg_rev, x='month', y='total_revenue', hue='plan', marker='o', ax=axes[1])
axes[1].set_title('Average Monthly Revenue by Plan')
axes[1].set_ylabel('Avg Revenue (USD)')

plt.tight_layout()
plt.show()

user_usage.groupby('plan')['total_revenue'].agg(['mean','median','var','min','max']).round(2)

**Finding:** Ultimate generates higher and more stable revenue (avg $72.31, variance = 129). Surf revenue is lower on average ($60.71) but highly variable (variance = 3,067) due to unpredictable overage charges — some Surf users pay up to $590/month.

## 6. Hypothesis Testing

### Test 1 — Do Surf and Ultimate generate different average revenue?

- **H₀:** Average monthly revenue is equal for both plans
- **H₁:** Average monthly revenue differs between plans
- **α = 0.05**

In [ ]:
surf_rev     = user_usage[user_usage['plan'] == 'surf']['total_revenue']
ultimate_rev = user_usage[user_usage['plan'] == 'ultimate']['total_revenue']

alpha = 0.05

# Levene's test for equal variances
lev_stat, lev_p = stats.levene(surf_rev, ultimate_rev)
equal_var = lev_p >= alpha
print(f'Levene p-value: {lev_p:.2e} → equal_var = {equal_var}')

# Independent t-test
t_stat, t_p = stats.ttest_ind(surf_rev, ultimate_rev, equal_var=equal_var)
print(f'T-test p-value: {t_p:.2e}')

if t_p < alpha:
    print('✅ Reject H₀ — Surf and Ultimate generate significantly different average revenue.')
else:
    print('❌ Fail to reject H₀ — no significant difference found.')

### Test 2 — Do NY-NJ users generate different average revenue than other regions?

- **H₀:** Average revenue is equal between NY-NJ and other regions
- **H₁:** Average revenue differs between NY-NJ and other regions
- **α = 0.05**

In [ ]:
ny_nj_mask   = user_usage['city'].astype(str).str.contains('NY-NJ', case=False, na=False)
ny_nj_rev    = user_usage.loc[ny_nj_mask,  'total_revenue'].dropna()
other_rev    = user_usage.loc[~ny_nj_mask, 'total_revenue'].dropna()

print(f'NY-NJ: N={len(ny_nj_rev)}, Mean=${ny_nj_rev.mean():.2f}')
print(f'Other: N={len(other_rev)}, Mean=${other_rev.mean():.2f}')

lev_stat2, lev_p2 = stats.levene(ny_nj_rev, other_rev)
equal_var2 = lev_p2 >= alpha
print(f'Levene p-value: {lev_p2:.6f} → equal_var = {equal_var2}')

t_stat2, t_p2 = stats.ttest_ind(ny_nj_rev, other_rev, equal_var=equal_var2)
print(f'T-test p-value: {t_p2:.6f}')

if t_p2 < alpha:
    print('✅ Reject H₀ — NY-NJ revenue differs significantly from other regions.')
else:
    print('❌ Fail to reject H₀ — no significant regional difference found.')

## 7. Conclusion

| Analysis | Finding |
|---|---|
| Plan revenue | **Ultimate generates 19% more revenue per user** ($72.31 vs $60.71) |
| Revenue stability | Ultimate variance = 129 vs Surf variance = 3,067 — Ultimate is far more predictable |
| Surf overage risk | Some Surf users pay up to $590/month — overage charges are significant |
| Regional difference | NY-NJ users generate slightly less revenue ($59.92 vs $65.22), p = 0.044 |
| Plan test | p < 0.0001 → revenue difference between plans is statistically significant |

**Strategic recommendation:**
- **Ultimate** should be the focus for revenue maximization campaigns — higher ARPU and predictable cash flow
- **Surf** drives customer acquisition volume — high overage potential makes it financially interesting despite lower base fee
- Consider targeted upsell campaigns for heavy Surf users who regularly exceed plan limits